In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:85% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:16pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))

In [6]:
import os
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
plt.rc('font', family='Malgun Gothic')
plt.rcParams['axes.unicode_minus'] = False
import warnings
warnings.filterwarnings('ignore')

## 1. 데이터와 모델 불러오기

In [7]:
df = pd.read_csv('C:/ai/lecNote/09_project/00_data/dataset_features.csv')
df.shape

(254750, 164)

In [8]:
m1_model = joblib.load('C:/ai/lecNote/09_project/M1_웹페이지/07_web/model/M1_y1_XGBoost.joblib')
m1_model.feature_names_in_   # 최종 모델이 학습한 변수 12개

array(['전체점포수_상대', '개업률_당분기', '업종밀집도', '순증가율', '폐업률_당분기', '유동인구_log',
       '프랜차이즈_비율', '점포당매출_log', '업종', '요일매출_편차', '총매출_log', '영역면적_log'],
      dtype='<U9')

## 2. 웹용 컬럼 선택

- 화면 표시용 컬럼 + 모델 변수 12개

In [10]:
web_data = df[['기준_년분기_코드', '자치구_코드_명', '행정동_코드_명', '상권_코드', '상권_코드_명',
               '상권_구분_코드_명', '서비스_업종_코드_명', '위도', '경도',
               '전체_점포_수', '당월_매출_금액', '총_유동인구_수',
               '전체점포수_상대', '개업률_당분기', '업종밀집도', '순증가율', '폐업률_당분기', '유동인구_log',
               '프랜차이즈_비율', '점포당매출_log', '업종', '요일매출_편차', '총매출_log', '영역면적_log']].copy()
web_data.shape

(254750, 24)

## 3. 화면 표시용 컬럼 추가

- 점포당 매출 : 분기 매출 ÷ 전체 점포 수, 만원 단위 (점포당매출_log와 같은 정의)
- 매출은 왜도가 커서(73) 평균이 큰 상권에 끌려가므로 **업종 중앙값**과 비교
- 폐업률은 종속변수 정의와 같게 **업종 평균**과 비교

In [11]:
web_data['점포당매출_만원'] = web_data['당월_매출_금액'] / web_data['전체_점포_수'] / 10000
web_data['점포당매출_만원'].describe().round(0)

count    254750.0
mean       6273.0
std       15623.0
min           0.0
25%         784.0
50%        2372.0
75%        6472.0
max      780410.0
Name: 점포당매출_만원, dtype: float64

In [12]:
web_data['점포당매출_업종중앙값'] = web_data.groupby(['기준_년분기_코드', '서비스_업종_코드_명'])['점포당매출_만원'].transform('median')

In [13]:
web_data['폐업률_업종평균'] = web_data.groupby(['기준_년분기_코드', '서비스_업종_코드_명'])['폐업률_당분기'].transform('mean')

In [14]:
web_data.head()

,기준_년분기_코드,자치구_코드_명,행정동_코드_명,상권_코드,상권_코드_명,상권_구분_코드_명,서비스_업종_코드_명,위도,경도,전체_점포_수,당월_매출_금액,총_유동인구_수,전체점포수_상대,개업률_당분기,업종밀집도,순증가율,폐업률_당분기,유동인구_log,프랜차이즈_비율,점포당매출_log,업종,요일매출_편차,총매출_log,영역면적_log,점포당매출_만원,점포당매출_업종중앙값,폐업률_업종평균
0,20231,용산구,이태원1동,3001491,이태원 관광특구,관광특구,한식음식점,37.5343,126.99432,126,8510268369,2014294.0,3.645280,4,0.625529,0.000000,4,14.515780,0.071429,18.028257,57,0.039950,22.864539,12.828169,6754.181245,4611.894557,3.691489
1,20232,용산구,이태원1동,3001491,이태원 관광특구,관광특구,한식음식점,37.5343,126.99432,128,10471475721,2169881.0,3.714415,2,0.589894,0.015625,1,14.590183,0.070312,18.219891,57,0.045287,23.071921,12.828169,8180.840407,5009.851466,3.922096
2,20233,용산구,이태원1동,3001491,이태원 관광특구,관광특구,한식음식점,37.5343,126.99432,127,11852127714,2131046.0,3.680666,3,0.595951,-0.007874,4,14.572124,0.078740,18.351586,57,0.063527,23.195773,12.828169,9332.384027,4901.244860,4.123132
3,20234,용산구,이태원1동,3001491,이태원 관광특구,관광특구,한식음식점,37.5343,126.99432,128,12822193259,2116566.0,3.687329,2,0.604753,0.007812,1,14.565306,0.078125,18.422413,57,0.050142,23.274443,12.828169,10017.338484,5060.787067,3.609408
4,20241,용산구,이태원1동,3001491,이태원 관광특구,관광특구,한식음식점,37.5343,126.99432,134,11857508645,2155176.0,3.795313,8,0.621759,0.044776,3,14.583383,0.089552,18.298387,57,0.058389,23.196227,12.828169,8848.887049,4767.490106,3.647646


## 4. 예측 확인

- predict.py에서 하는 예측을 미리 실행해 행 수와 위기확률 분포 확인 (2025년 4분기 21,333행)

In [15]:
pred_check = web_data[web_data['기준_년분기_코드'] == 20254]
pred_check.shape

(21333, 27)

In [16]:
위기확률_check = m1_model.predict_proba(pred_check[m1_model.feature_names_in_])[:, 1]
pd.Series(위기확률_check).describe().round(3)

count    21333.000
mean         0.418
std          0.238
min          0.004
25%          0.217
50%          0.414
75%          0.623
max          0.928
dtype: float64

In [18]:
최종기준 = 0.55   # 05-3 노트북 5.1에서 구한 값으로 바꾸기
pd.Series(np.where(위기확률_check >= 최종기준, '위기', '양호')).value_counts()

양호    14306
위기     7027
dtype: int64

## 5. 저장

In [19]:
os.makedirs('C:/ai/lecNote/09_project/M1_웹페이지/07_web/data', exist_ok=True)

In [20]:
web_data.to_csv('C:/ai/lecNote/09_project/M1_웹페이지/07_web/data/web_data.csv', index=False, encoding='utf-8-sig')